# Exploratory Data Analysis (EDA) & Encoding - E4
Mainly, this notebook covers EDA on the raw trace data and encoding it for use in other steps of the development process.
Also this notebook will compare the performance of using the both method, Which is more robust to go forward without losing any data in the process

In [2]:
import sys
import os
import time
import json
import pandas as pd
import numpy as np
from collections import Counter

sys.path.append("..")

# Import the function from the utils which will needed here
from utils import (
    stream_opcodes,
    count_opcodes_from_stream,
    build_opcode_vocab,
    encode_trace_to_npy,
    save_vocab,
    load_vocab,
    check_vocab_size,
    decode_trace_npy,
    load_encoded_trace
)

## Initializing the Paths

In [3]:
RAW_TRACES_DIR = "../sandbox/data-e4/raw_traces"
MANIFEST_CSV = "../sandbox/data-e4/manifest.csv"

# Load the manifest CSV to get the metadata to use later
df_manifest = pd.read_csv(MANIFEST_CSV)
print(f"Manifest Loaded: {len(df_manifest)} rows")

# Get the list of raw trace files in the directory and sort them for processing
trace_files = sorted([
    os.path.join(RAW_TRACES_DIR, f)
    for f in os.listdir(RAW_TRACES_DIR)
    if f.endswith(".txt")
])
print(f"Raw trace files found: {len(trace_files)}")

Manifest Loaded: 260 rows
Raw trace files found: 100


----
## EDA using Raw Trace Streaming - Method 1
Each of the raw opcode trace file is streamed line by line using `stream_opcodes()` function. A single pass over each file simultaneously to improve the performance.

In [4]:
# Set the data structures to hold the global vocab, per file opcode counts, and streaming times
global_vocab = set()
per_file_counts = {}
stream_times = {}

MIN_OPCODE_COUNT = 10 # Filter threshold to removes likely malformed or truncated opcodes

print(f"Streaming {len(trace_files)} trace files...")
print("This will take a moment base on the given file sizes. The progress shown per 10 files\n")

total_start = time.time()

# Stream through each trace file count the opcodes and store the counts and elapsed time
for i, path in enumerate(trace_files, 1):
    filename = os.path.basename(path)
    t0 = time.time()

    counter = Counter()
    for opcode in stream_opcodes(path):
        counter[opcode] += 1

    elapsed = time.time() - t0
    stream_times[filename] = elapsed
    per_file_counts[filename] = counter

    if i % 10 == 0 or i == len(trace_files):
        done = time.time() - total_start
        print(f"  [{i:>3}/{len(trace_files)}]  elapsed: {done/60:.1f} min")

# Build global counts then filter by minimum threshold
global_counter = Counter()
for c in per_file_counts.values():
    global_counter.update(c)

# Filter the global counter to remove opcodes that are below the minimum threshold
removed = [op for op, count in global_counter.items() if count < MIN_OPCODE_COUNT]
global_vocab = {op for op, count in global_counter.items() if count >= MIN_OPCODE_COUNT}

total_stream_time = time.time() - total_start
print(f"\nStreaming Complete:  {total_stream_time/60:.1f} min total")
print(f"Raw Vocabulary Size: {len(global_counter)} distinct opcodes")
print(f"After Filtering (>={MIN_OPCODE_COUNT}): {len(global_vocab)} distinct opcodes")
print(f"Removed {len(removed)}")

Streaming 100 trace files...
This will take a moment base on the given file sizes. The progress shown per 10 files

  [ 10/100]  elapsed: 16.5 min
  [ 20/100]  elapsed: 30.9 min
  [ 30/100]  elapsed: 44.7 min
  [ 40/100]  elapsed: 61.7 min
  [ 50/100]  elapsed: 75.4 min
  [ 60/100]  elapsed: 88.3 min
  [ 70/100]  elapsed: 103.9 min
  [ 80/100]  elapsed: 119.1 min
  [ 90/100]  elapsed: 132.6 min
  [100/100]  elapsed: 146.4 min

Streaming Complete:  146.4 min total
Raw Vocabulary Size: 806243 distinct opcodes
After Filtering (>=10): 87244 distinct opcodes
Removed 718999


### Opcode Vocabulary
The full set of distinct x86 opcode mnemonics seen across all 260 files. `check_vocab_size` confirms the count is safely within the `int32` range used for encoding.

In [5]:
vocab = sorted(global_vocab)
check_vocab_size(vocab)

print(f"Vocabulary Size: {len(vocab)} distinct opcodes")
# print(f"Opcodes List: {vocab}")

Vocabulary Size: 87244 distinct opcodes


### Crosscheck opcode counts with the manifest data

The manifest already has `opcode_count` from the log's "Opcodes recorded" line. Comparing these with the actual line counts from the streaming pass confirms the raw trace files are complete and uncorrupted.

In [6]:
stream_counts = {
    fname: sum(c.values())
    for fname, c in per_file_counts.items()
}

df_cross = df_manifest[["filename", "opcode_count"]].copy()
df_cross["stream_count"] = df_cross["filename"].map(stream_counts)
df_cross["match"] = df_cross["opcode_count"] == df_cross["stream_count"]

mismatches = df_cross[~df_cross["match"].fillna(False)]
print(f"Files with matching counts: {df_cross['match'].sum()}")
print(f"Mismatches or missing count: {len(mismatches)}")

if len(mismatches):
    print("\nMismatched Files:")
    print(mismatches)

Files with matching counts: 0
Mismatches or missing count: 260

Mismatched Files:
                                              filename  opcode_count  \
0    GommeHD.net-Java-API-2.2.0-jar-with-dependenci...     263246339   
1                                     abbot-0.13.0.jar     250721862   
2                              activecluster-4.0.2.jar     192722442   
3                                  activemq-4.0-M1.jar     253984851   
4                               activemq-all-6.2.6.jar     249490122   
..                                                 ...           ...   
255                         xercesjarv-20020712-DG.jar     142176262   
256                               xml-resolver-1.2.jar     190981714   
257                                      xom-1.4.5.jar     184326583   
258                                xsdlib-20050913.jar     199471927   
259                                  ymsg-test-0.6.jar     216474789   

     stream_count  match  
0             NaN  False  

### Global Opcode Frequency Distribution

Analyze the frequency across all 260 files, which opcodes dominate the instruction mix. This clearly shows later feature engineering decisions.

This is a summary table only for this stage, plots will be created on later step.

In [7]:
global_counter = Counter()

for count in per_file_counts.values():
    global_counter.update(count)

df_freq = pd.DataFrame(
    global_counter.most_common(),
    columns=["opcode", "total_count"]
)

df_freq["proportion"] = df_freq["total_count"] / df_freq["total_count"].sum()

print(f"Top 20 most frequent opcodes across all {len(trace_files)} files:\n")
print(df_freq.head(20).to_string(index=False))

Top 20 most frequent opcodes across all 100 files:

   opcode  total_count  proportion
      MOV   6740331992    0.311678
      CMP   1710524561    0.079096
       JZ   1197168401    0.055358
      ADD   1185160645    0.054803
     PUSH   1084309761    0.050139
     TEST   1028402898    0.047554
      POP    815426884    0.037706
      LEA    753222809    0.034830
      JNZ    729689629    0.033741
    MOVZX    626040209    0.028949
      JMP    571720994    0.026437
      NOP    422428107    0.019533
      AND    408022294    0.018867
      SUB    397036851    0.018359
 RET_NEAR    355665925    0.016446
CALL_NEAR    350068079    0.016187
   MOVSXD    259561950    0.012002
    STOSB    254645735    0.011775
      SHR    234598937    0.010848
      XOR    227629026    0.010526


#### Bottom of the distribution to list the rarest opcodes

In [8]:
print("Rarest 20 opcodes (lowest global frequency):\n")
print(df_freq.tail(20).to_string(index=False))

Rarest 20 opcodes (lowest global frequency):

              opcode  total_count   proportion
             EASTJMP            1 4.624076e-11
      MOVVSUBOUPPERU            1 4.624076e-11
        MOVJBMOVCMPO            1 4.624076e-11
     PUSHMOJMPL_NEAR            1 4.624076e-11
         MOVMOJLEPOP            1 4.624076e-11
    MOVCANOPJPUSHMOV            1 4.624076e-11
          OPOPCMPJMP            1 4.624076e-11
CMPXOADDJNLERET_NEAR            1 4.624076e-11
               SOVDZ            1 4.624076e-11
               LE_NE            1 4.624076e-11
       JNLCALL_NEARL            1 4.624076e-11
         UMOVMOVVMOV            1 4.624076e-11
       SUBCALL_NEAPR            1 4.624076e-11
                 CQP            1 4.624076e-11
            JNBELEOV            1 4.624076e-11
  CALL_NEARPUSHMJMPM            1 4.624076e-11
         SLCALL_NEAR            1 4.624076e-11
         PUSHANDZMOV            1 4.624076e-11
  JNLEPUNPCKLQDQRMOV            1 4.624076e-11
        JNBSCM

### Check per file trace length distribution

How much do the 260 files vary in total opcode counts, large variation matters for normalization and some file have 260M opcodes and some are 50M opcodes, this should contribute comparable feature vectors, not be dominated by raw size.

In [9]:
df_lengths = pd.DataFrame({
    "filename": list(stream_counts.keys()),
    "opcode_count": list(stream_counts.values()),
})

print(df_lengths["opcode_count"].describe().apply(lambda x: f"{x:,.0f}"))

count            100
mean     216,259,410
std       45,288,333
min      133,072,452
25%      173,500,454
50%      246,227,151
75%      252,807,844
max      274,216,462
Name: opcode_count, dtype: str


### Streaming Timing Summary

Time taken per file using raw text streaming, the baseline for the performance comparison with Method 2 (Encoded Array) later.

In [10]:
df_timing = pd.DataFrame({
    "filename": list(stream_times.keys()),
    "stream_sec": list(stream_times.values()),
})

df_timing["opcodes"] = df_timing["filename"].map(stream_counts)
df_timing["lines_per_sec"] = (df_timing["opcodes"] / df_timing["stream_sec"]).astype(int)

print(f"Streaming Speed Summary (Method 1)")
print(df_timing["lines_per_sec"].describe().apply(lambda x: f"{x:,.0f}"))
print(f"\nTotal Streaming Time: {df_timing['stream_sec'].sum()/60:.1f} min")

Streaming Speed Summary (Method 1)
count          100
mean     2,467,073
std         70,854
min      2,149,301
25%      2,434,377
50%      2,469,781
75%      2,502,678
max      2,609,553
Name: lines_per_sec, dtype: str

Total Streaming Time: 146.4 min


### Save EDA Outputs into CSVs

Save the frequency table and timing results as CSVs so it can be used for later visualization and later notebooks can load them without re-streaming.

In [11]:
os.makedirs("../sandbox/data-e4/processed_outputs", exist_ok=True)

df_freq.to_csv("../sandbox/data-e4/processed_outputs/global_opcode_freq.csv", index=False)
print("global_opcode_freq.csv file saved!")

df_lengths.to_csv("../sandbox/data-e4/processed_outputs/per_file_lengths.csv", index=False)
print("per_file_lengths.csv file saved!")

df_timing.to_csv("../sandbox/data-e4/processed_outputs/streaming_timing.csv", index=False)
print("streaming_timing.csv file saved!")

global_opcode_freq.csv file saved!
per_file_lengths.csv file saved!
streaming_timing.csv file saved!


----
### FIX: Crosscheck opcode counts with the manifest data
The cross-check mismatch is actually a filename extension bug, the manifest.csv has `.jar` filenames but stream_counts keys are `.txt` filenames, so .map() finds nothing. Not a data problem.

In [12]:
# Fix: strip .txt and add .jar to match manifest filename format
stream_counts_jar = {
    fname.replace(".txt", ".jar"): count
    for fname, count in stream_counts.items()
}

df_cross = df_manifest[["filename", "opcode_count"]].copy()
df_cross["stream_count"] = df_cross["filename"].map(stream_counts_jar)
df_cross["match"] = df_cross["opcode_count"] == df_cross["stream_count"]

mismatches = df_cross[~df_cross["match"].fillna(False)]
print(f"Files with matching counts: {df_cross['match'].sum()}")
print(f"Mismatches or missing: {len(mismatches)}")

if len(mismatches):
    print(f"\nMismatched File:\n{mismatches}")

Files with matching counts: 0
Mismatches or missing: 260

Mismatched File:
                                              filename  opcode_count  \
0    GommeHD.net-Java-API-2.2.0-jar-with-dependenci...     263246339   
1                                     abbot-0.13.0.jar     250721862   
2                              activecluster-4.0.2.jar     192722442   
3                                  activemq-4.0-M1.jar     253984851   
4                               activemq-all-6.2.6.jar     249490122   
..                                                 ...           ...   
255                         xercesjarv-20020712-DG.jar     142176262   
256                               xml-resolver-1.2.jar     190981714   
257                                      xom-1.4.5.jar     184326583   
258                                xsdlib-20050913.jar     199471927   
259                                  ymsg-test-0.6.jar     216474789   

     stream_count  match  
0             NaN  False  
1     

The fix actually worked - stream_count is now populated with real numbers, not NaN anymore. There is a tiny difference that is expected and not a problem. 
Because the manifest opcode_count comes from run_trace.sh which uses `wc -l` to count lines. So `wc -l` counts every new line character including empty lines. But the stream_opcodes function skips empty lines. So any empty lines in the .txt file get counted by `wc -l` but skipped by the streamer.

### Fix the crosscheck to use approximate matching instead of exact equal

In [13]:
# Allow a small tolerance value to account for empty lines
TOLERANCE = 0.001

df_cross["diff_pct"] = abs(df_cross["opcode_count"] - df_cross["stream_count"]) / df_cross["opcode_count"]
df_cross["match"] = df_cross["diff_pct"] <= TOLERANCE

mismatches = df_cross[~df_cross["match"].fillna(False)]
print(f"Files within {TOLERANCE*100}% Tolerance: {df_cross['match'].sum()}")
print(f"Files Outside Tolerance: {len(mismatches)}")
print(f"\nMax Difference across all files: {df_cross['diff_pct'].max()*100:.4f}%")
print(f"Mean Difference across all files: {df_cross['diff_pct'].mean()*100:.4f}%")

Files within 0.1% Tolerance: 3
Files Outside Tolerance: 257

Max Difference across all files: 19.7622%
Mean Difference across all files: 2.7949%


### E4 Vocabulary coverage check against the fixed E1 287-opcode vocab

In [14]:
# The vocab is not rebuilt for E2 - it stays fixed
# so feature vectors remain directly comparable across environments
E1_CLEAN_VOCAB = "../sandbox/data/vocab_clean.json"
clean_vocab = load_vocab(E1_CLEAN_VOCAB)
print(f"E1 clean vocab loaded: {len(clean_vocab)} opcodes")

covered = df_freq[df_freq.opcode.isin(clean_vocab)]
total_e2 = df_freq.total_count.sum()
kept_e2 = covered.total_count.sum()

print(f"\nE2 stream covered by fixed 287 vocab: {kept_e2/total_e2*100:.4f}%")
print(f"E2 stream not covered: {(total_e2-kept_e2)/total_e2*100:.4f}%")
print(f"Vocab Entries present in E2: {len(covered)} of {len(clean_vocab)}")

# Opcodes in the fixed vocab that never appear in E2
absent = sorted(set(clean_vocab) - set(covered.opcode))
print(f"\nVocab opcodes absent from E2 ({len(absent)}): {absent}")

# Highest-frequency E2 opcodes NOT in the fixed vocab
missing = df_freq[~df_freq.opcode.isin(clean_vocab)].head(30)
print(f"\nTop 30 E2 opcodes outside the fixed vocab:")
print(missing.to_string(index=False))

E1 clean vocab loaded: 287 opcodes

E2 stream covered by fixed 287 vocab: 99.7712%
E2 stream not covered: 0.2288%
Vocab Entries present in E2: 274 of 287

Vocab opcodes absent from E2 (13): ['CMPSW', 'LFENCE', 'MOVNTDQ', 'MOVNTPS', 'PADDB', 'RCR', 'SETNO', 'SFENCE', 'SHUFPS', 'VMOVLPD', 'VMOVNTDQ', 'VPCMPEQQ', 'VPUNPCKHQDQ']

Top 30 E2 opcodes outside the fixed vocab:
 opcode  total_count  proportion
     OV      1933669    0.000089
 MOVMOV      1642061    0.000076
      V       868507    0.000040
     MP       685474    0.000032
     MO       585059    0.000027
      P       559437    0.000026
      M       455471    0.000021
      Z       421609    0.000019
 MOVCMP       396112    0.000018
 CMPMOV       395693    0.000018
     OP       395101    0.000018
    USH       307483    0.000014
    EST       286733    0.000013
     DD       281816    0.000013
PUSHMOV       271386    0.000013
  JZMOV       265967    0.000012
   MMOV       259470    0.000012
MOVPUSH       256479    0.000012
  